# Лабораторная работа №6. Вариант 7
## Квантовые алгоритмы Дойча, Дойча–Йожи и Шора (Qiskit 2.x)

**Параметры варианта 7:**
* **Задание 1 (алгоритм Дойча):** $7 \equiv 2 \pmod 5 \Rightarrow$ в пунктах 1.1–1.2 используется `twobit_function(2)` ($f_2(x) = x$, сбалансированная); пункты 1.3–1.5 выполняются для всех четырёх функций.
* **Задание 2 (алгоритм Дойча–Йожи):** $7 \equiv 2 \pmod 5 \Rightarrow$ число входных кубитов $n = 3$ ($\ge 5$ случайных запусков); пункты 2.2–2.4 выполняются для всех вариантов.
* **Задания 3–4 (алгоритм Шора, Таблица А):** $N = 35$, $a = 2$, разрядность $n = \lceil \log_2 35 \rceil = 6$, число управляющих кубитов $m = 2n = 12$ (всего $3n = 18$ кубитов).
* **Задание 5:** относится только к вариантам 16–30 (Таблица Б) и для варианта 7 не выполняется.
* **Задание 6:** универсальная функция факторизации `shor_factor(N, bases)` и ответы на контрольные вопросы.

In [ ]:
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
import sympy

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2
from qiskit.quantum_info import Statevector, partial_trace
from qiskit.circuit.library import UnitaryGate

backend_sim = AerSimulator()
sampler = BackendSamplerV2(backend=backend_sim)

def run_counts(qc: QuantumCircuit, shots: int = 1024, seed: int | None = None) -> dict[str, int]:
    if seed is None:
        job = sampler.run([qc], shots=shots)
    else:
        seeded_sampler = BackendSamplerV2(backend=AerSimulator(seed_simulator=seed))
        job = seeded_sampler.run([qc], shots=shots)
    pub_result = job.result()[0]
    creg_data = next(iter(pub_result.data.values()))
    return creg_data.get_counts()

---
## Задание 1. Алгоритм Дойча

### Пункты 1.1–1.2. Запуск схемы для `twobit_function(2)` и анализ фазового отката

После подготовки вспомогательного кубита 1 в состоянии $|-
angle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$ и входного кубита 0 в состоянии $|+
angle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ действие оракула $U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$ приводит систему к состоянию:
$$|\psi\rangle = (-1)^{f(0)} \frac{|0\rangle_0 + (-1)^{f(0)\oplus f(1)}|1\rangle_0}{\sqrt{2}} |-\rangle_1.$$
Для сбалансированной функции ($f_2(0) = 0 \neq f_2(1) = 1$) кубит 0 за счёт фазового отката (*phase kickback*) переходит в состояние $|-
angle$, а заключительный вентиль $H$ переводит $H|-\rangle = |1\rangle$, поэтому измерение детерминированно даёт `1`.

In [ ]:
def twobit_function(case: int) -> QuantumCircuit:
    if case not in (1, 2, 3, 4):
        raise ValueError("`case` должен быть 1, 2, 3 или 4.")
    f = QuantumCircuit(2)
    if case in (2, 3):
        f.cx(0, 1)
    if case in (3, 4):
        f.x(1)
    return f

def classical_type_1bit(case: int) -> str:
    return {1: "постоянная", 2: "сбалансированная", 3: "сбалансированная", 4: "постоянная"}[case]

# 1.1. Схема для case = 2 (вариант 7: 7 mod 5 = 2)
qc_1_1 = QuantumCircuit(2, 1)
qc_1_1.x(1)
qc_1_1.h([0, 1])
qc_1_1.barrier()
qc_1_1.compose(twobit_function(2), inplace=True)
qc_1_1.barrier()
qc_1_1.h(0)
qc_1_1.measure(0, 0)

counts_1_1 = run_counts(qc_1_1, shots=1024)
measured_bit = max(counts_1_1, key=counts_1_1.get)
quantum_ans = "сбалансированная" if measured_bit == "1" else "постоянная"
print("Counts (case=2):", counts_1_1)
print("Квантовый ответ:", quantum_ans)
assert quantum_ans == classical_type_1bit(2)

# 1.2. Редуцированное состояние кубита 0 после оракула
sv_plus = Statevector.from_label("+")
sv_minus = Statevector.from_label("-")
for case in (1, 2, 3, 4):
    qc_sv = QuantumCircuit(2)
    qc_sv.x(1)
    qc_sv.h([0, 1])
    qc_sv.compose(twobit_function(case), inplace=True)
    psi_0 = partial_trace(Statevector.from_instruction(qc_sv), [1]).to_statevector()
    actual = "|+>" if np.allclose(np.abs(np.vdot(psi_0, sv_plus)), 1.0) else "|->"
    expected = "|+>" if case in (1, 4) else "|->"
    print(f"Функция {case} ({classical_type_1bit(case)}): состояние кубита 0 = {actual}")
    assert actual == expected

### Пункты 1.3–1.5. Эксперименты с удалением вентилей и чёрный ящик `deutsch_decide`

* **1.3 (без `X` на кубите 1):** целевой кубит остаётся в собственном состоянии $|+
angle$ оператора $X$ ($X|+\rangle = |+\rangle$), фазовый откат не возникает, и для всех 4 функций измеряется `0` («постоянная»).
* **1.4 (без первого `H` на кубите 0):** оракул вычисляет функцию только в точке $x=0$, а заключительный $H|0\rangle = |+\rangle$ даёт случайные `0` и `1` с вероятностью $50\%$ для всех 4 функций.

In [ ]:
print("=== Пункт 1.3: без вентиля X на целевом кубите 1 ===")
for case in (1, 2, 3, 4):
    qc_no_x = QuantumCircuit(2, 1)
    qc_no_x.h([0, 1])
    qc_no_x.compose(twobit_function(case), inplace=True)
    qc_no_x.h(0)
    qc_no_x.measure(0, 0)
    counts = run_counts(qc_no_x, shots=1024)
    res = "сбалансированная" if max(counts, key=counts.get) == "1" else "постоянная"
    print(f"f_{case}: counts={counts} -> '{res}' (верный ответ: '{classical_type_1bit(case)}')")

print("\n=== Пункт 1.4: без первого вентиля H на управляющем кубите 0 ===")
for case in (1, 2, 3, 4):
    qc_no_h0 = QuantumCircuit(2, 1)
    qc_no_h0.x(1)
    qc_no_h0.h(1)
    qc_no_h0.compose(twobit_function(case), inplace=True)
    qc_no_h0.h(0)
    qc_no_h0.measure(0, 0)
    counts = run_counts(qc_no_h0, shots=1024)
    print(f"f_{case}: counts={counts} (верный ответ: '{classical_type_1bit(case)}')")

def deutsch_decide(oracle: QuantumCircuit) -> str:
    gate = oracle.to_gate()
    gate.name = "U_f"
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h([0, 1])
    qc.append(gate, [0, 1])
    qc.h(0)
    qc.measure(0, 0)
    assert qc.count_ops().get("U_f", 0) == 1
    counts = run_counts(qc, shots=128)
    return "сбалансированная" if max(counts, key=counts.get) == "1" else "постоянная"

rng = np.random.default_rng(7)
for case in (1, 2, 3, 4):
    assert deutsch_decide(twobit_function(case)) == classical_type_1bit(case)
for _ in range(100):
    case = int(rng.integers(1, 5))
    oracle = twobit_function(case)
    f0 = int(np.argmax(np.abs(Statevector.from_label("00").evolve(oracle).data)) >> 1)
    f1 = int(np.argmax(np.abs(Statevector.from_label("01").evolve(oracle).data)) >> 1)
    assert deutsch_decide(oracle) == ("постоянная" if f0 == f1 else "сбалансированная")
print("\nПункт 1.5: все 104 теста deutsch_decide пройдены успешно!")

---
## Задание 2. Алгоритм Дойча–Йожи

### Пункты 2.1–2.3. Случайные оракулы ($n = 3$), синтез по таблице истинности и нарушение обещания

При $k$ единицах в таблице истинности из $2^n$ значений вероятность получить на выходе строку из нулей равна:
$$P(00\ldots0) = \left( \frac{1}{2^n} \sum_{x=0}^{2^n - 1} (-1)^{f(x)} \right)^2 = \left( 1 - \frac{k}{2^{n-1}} \right)^2.$$
Для $n = 3$: при $k = 1 \Rightarrow P(000) = 0.5625$; при $k = 2 \Rightarrow P(000) = 0.2500$; при $k = 3 \Rightarrow P(000) = 0.0625$. Поскольку $P(000) > 0$, одиночный замер `000` не гарантирует постоянства функции, если не выполняется обещание алгоритма Дойча–Йожи.

In [ ]:
def dj_function(n: int, rng: np.random.Generator | None = None) -> QuantumCircuit:
    if rng is None:
        rng = np.random.default_rng()
    qc = QuantumCircuit(n + 1)
    if not bool(rng.integers(0, 2)):
        if rng.integers(0, 2):
            qc.x(n)
    else:
        mask = int(rng.integers(1, 2**n))
        for i in range(n):
            if (mask >> i) & 1:
                qc.cx(i, n)
        if rng.integers(0, 2):
            qc.x(n)
    return qc

def true_type_by_truth_table(oracle: QuantumCircuit, n: int) -> tuple[str, str]:
    outputs = []
    for x in range(2**n):
        qc_test = QuantumCircuit(n + 1, 1)
        for i in range(n):
            if (x >> i) & 1:
                qc_test.x(i)
        qc_test.compose(oracle, inplace=True)
        qc_test.measure(n, 0)
        counts = run_counts(qc_test, shots=1)
        outputs.append(max(counts, key=counts.get))
    bitstring = "".join(outputs)
    ones = outputs.count("1")
    ftype = "постоянная" if ones in (0, 2**n) else ("сбалансированная" if ones == 2**(n - 1) else "иная")
    return bitstring, ftype

def run_dj(oracle: QuantumCircuit, n: int, shots: int = 1024) -> str:
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.compose(oracle, inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    counts = run_counts(qc, shots=shots)
    return "постоянная" if max(counts, key=counts.get) == "0" * n else "сбалансированная"

def dj_from_truth_table(f_vals: list[int]) -> QuantumCircuit:
    size = len(f_vals)
    n = int(round(math.log2(size)))
    assert 2**n == size
    qc = QuantumCircuit(n + 1)
    for x, val in enumerate(f_vals):
        if val == 1:
            zeros = [i for i in range(n) if not ((x >> i) & 1)]
            if zeros:
                qc.x(zeros)
            qc.mcx(list(range(n)), n)
            if zeros:
                qc.x(zeros)
    return qc

# 2.1. 6 случайных запусков для n = 3 (вариант 7: 7 mod 5 = 2)
rng = np.random.default_rng(7)
for idx in range(1, 7):
    oracle = dj_function(3, rng)
    bits, truth = true_type_by_truth_table(oracle, 3)
    ans = run_dj(oracle, 3, shots=1024)
    assert ans == truth
    print(f"Запуск {idx}: таблица={bits}, алгоритм='{ans}', истина='{truth}'")

# 2.2. 300 тестов синтеза из таблицы истинности (n = 2, 3, 4)
rng_tt = np.random.default_rng(707)
for n_val in (2, 3, 4):
    n_in = 2**n_val
    for _ in range(50):
        assert run_dj(dj_from_truth_table([int(rng_tt.integers(0, 2))] * n_in), n_val, shots=64) == "постоянная"
    for _ in range(50):
        f_vals = [0] * (n_in // 2) + [1] * (n_in // 2)
        rng_tt.shuffle(f_vals)
        assert run_dj(dj_from_truth_table(f_vals), n_val, shots=64) == "сбалансированная"
print("Пункт 2.2: 300 тестов dj_from_truth_table пройдены успешно.")

# 2.3. Нарушение обещания (1, 2, 3 единицы при n = 3)
for k in (1, 2, 3):
    qc = QuantumCircuit(4, 3)
    qc.x(3)
    qc.h(range(4))
    qc.compose(dj_from_truth_table([1] * k + [0] * (8 - k)), inplace=True)
    qc.h(range(3))
    qc.measure(range(3), range(3))
    p_sim = run_counts(qc, shots=20000).get("000", 0) / 20000
    print(f"k = {k}: теория P(000) = {((8 - 2*k)/8)**2:.4f}, симуляция = {p_sim:.4f}")

### Пункт 2.4. Масштабирование ($n = 2 \dots 7$) и сравнение подходов

In [ ]:
rng_scale = np.random.default_rng(77)
ns = list(range(2, 8))
det_queries, prob_queries, quant_queries = [], [], []

for n_val in ns:
    oracle = dj_function(n_val, rng_scale)
    _, truth = true_type_by_truth_table(oracle, n_val)
    qc = QuantumCircuit(n_val + 1, n_val)
    qc.x(n_val)
    qc.h(range(n_val + 1))
    qc.compose(oracle, inplace=True)
    qc.h(range(n_val))
    qc.measure(range(n_val), range(n_val))

    t_qc = transpile(qc, backend_sim)
    counts = run_counts(t_qc, shots=256)
    ans = "постоянная" if max(counts, key=counts.get) == "0" * n_val else "сбалансированная"
    assert ans == truth

    d_q = 2 ** (n_val - 1) + 1
    det_queries.append(d_q)
    quant_queries.append(1)
    for q in range(1, d_q + 1):
        p_err = 0.0 if q > 2 ** (n_val - 1) else 2 * math.comb(2 ** (n_val - 1), q) / math.comb(2**n_val, q)
        if p_err <= 0.01:
            prob_queries.append(q)
            break
    print(f"n={n_val}: кубитов={n_val+1}, глубина={t_qc.depth()}, дет.={d_q}, вер.(P_err<=0.01)={prob_queries[-1]}")

balanced_arr = np.array([0] * 4 + [1] * 4)
trials = 100_000
errors = sum(1 for _ in range(trials) if np.all((s := rng_scale.choice(balanced_arr, size=3, replace=False)) == s[0]))
print(f"Монте-Карло (n=3, q=3): оценка = {errors/trials:.5f}, точная формула = {2*math.comb(4,3)/math.comb(8,3):.5f}")

plt.figure(figsize=(8, 5))
plt.plot(ns, det_queries, marker="o", label="Классический детерминированный ($2^{n-1}+1$)")
plt.plot(ns, prob_queries, marker="s", label="Классический вероятностный ($P_{err} \leq 0.01$)")
plt.plot(ns, quant_queries, marker="^", label="Квантовый Дойча–Йожи ($1$ запрос)")
plt.xlabel("Число входных кубитов n")
plt.ylabel("Число запросов к оракулу")
plt.title("Сравнение запросной сложности для задачи Дойча–Йожи")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("dj_complexity_variant_7.png", dpi=150)
plt.show()

---
## Задания 3, 4 и 6. Алгоритм Шора (вариант 7: $N = 35, a = 2$)

* $\gcd(2, 35) = 1$, разложение $35 = 5 \cdot 7$.
* Наименьший показатель $r > 0$, для которого $2^r \equiv 1 \pmod{35}$, равен **$r = 12$** ($2^{12} = 4096 = 117 \cdot 35 + 1$).
* Функция Эйлера $\varphi(12) = 4$ (взаимно просты с $12$ числители $k \in \{1, 5, 7, 11\}$), теоретическая доля точных восстановлений порядка равна $\varphi(r)/r = 4/12 = 1/3 \approx 0.3333$.
* Для $r = 12$: $x = 2^6 \bmod 35 = 64 \bmod 35 = 29$, откуда $d_1 = \gcd(28, 35) = 7$, $d_2 = \gcd(30, 35) = 5$.
* Минимальное число повторов $T$ для вероятности успеха $\ge 0.99$ при $p = 1/3$: $1 - (2/3)^T \ge 0.99 \Rightarrow T = \lceil \ln(0.01)/\ln(2/3) \rceil = 12$.

In [ ]:
def inverse_qft(n_qubits: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits, name="IQFT")
    for i in range(n_qubits // 2):
        qc.swap(i, n_qubits - 1 - i)
    for j in range(n_qubits):
        for k in range(j):
            qc.cp(-np.pi / (2 ** (j - k)), k, j)
        qc.h(j)
    return qc

def mod_mult_gate(b: int, N: int, n_bits: int) -> UnitaryGate:
    dim = 2 ** n_bits
    U = np.zeros((dim, dim), dtype=complex)
    for x in range(N):
        U[(b * x) % N, x] = 1.0
    for x in range(N, dim):
        U[x, x] = 1.0
    for x in range(N):
        vec = np.zeros(dim, dtype=complex)
        vec[x] = 1.0
        expected = np.zeros(dim, dtype=complex)
        expected[(b * x) % N] = 1.0
        assert np.allclose(U @ vec, expected)
    return UnitaryGate(U, label=f"M_{b}")

def build_shor_order_circuit(N: int, a: int, m: int, n_bits: int) -> QuantumCircuit:
    qc = QuantumCircuit(m + n_bits, m)
    qc.x(m)
    qc.h(range(m))
    for k in range(m):
        b = pow(a, 2 ** k, N)
        if b != 1:
            qc.append(mod_mult_gate(b, N, n_bits).control(1), [k] + list(range(m, m + n_bits)))
    qc.compose(inverse_qft(m), qubits=range(m), inplace=True)
    qc.measure(range(m), range(m))
    return qc

N, a = 35, 2
n_bits = math.ceil(math.log2(N))  # 6
m = 2 * n_bits                    # 12
qc_shor = build_shor_order_circuit(N, a, m, n_bits)
t_shor = transpile(qc_shor, backend_sim)
counts_shor = run_counts(t_shor, shots=2000)

valid_shots = 0
found_orders: dict[int, int] = {}
for bs, cnt in counts_shor.items():
    rc = Fraction(int(bs, 2), 2 ** m).limit_denominator(N).denominator
    if rc > 1 and pow(a, rc, N) == 1:
        valid_shots += cnt
        found_orders[rc] = found_orders.get(rc, 0) + cnt

print(f"[3.3] Доля успешных измерений: {valid_shots / 2000:.4f} (теория phi(12)/12 = 0.3333)")

r_found = max(found_orders, key=found_orders.get)
x = pow(a, r_found // 2, N)
d1, d2 = math.gcd(x - 1, N), math.gcd(x + 1, N)
print(f"[3.4] r = {r_found}, x = {x}, делители: d1 = {d1}, d2 = {d2}")
assert d1 * d2 == N and 1 < d1 < N and 1 < d2 < N

print("\n[3.5] Зависимость доли успехов от m in {n+1 .. 2n}:")
for m_val in range(n_bits + 1, 2 * n_bits + 1):
    counts_m = run_counts(transpile(build_shor_order_circuit(N, a, m_val, n_bits), backend_sim), shots=2000)
    ok = sum(
        cnt for bs, cnt in counts_m.items()
        if (rc := Fraction(int(bs, 2), 2 ** m_val).limit_denominator(N).denominator) > 1 and pow(a, rc, N) == 1
    )
    print(f"  m = {m_val:2d}: доля успехов = {ok / 2000:.4f}")

# Задание 4: 5 независимых запусков
shares_4_1 = []
for seed in (107, 207, 307, 407, 507):
    res = run_counts(t_shor, shots=1024, seed=seed)
    ok = sum(
        cnt for bs, cnt in res.items()
        if (rc := Fraction(int(bs, 2), 2 ** m).limit_denominator(N).denominator) > 1 and pow(a, rc, N) == 1
    )
    shares_4_1.append(ok / 1024)

print(f"\n[4.1] Средняя доля (5 запусков): {np.mean(shares_4_1):.4f} ± {np.std(shares_4_1, ddof=1):.4f}")
print(f"[4.2] Минимальное число повторов T (P >= 0.99): T = {math.ceil(math.log(0.01) / math.log(1 - 4/12))}")

In [ ]:
def quantum_order_finding(N: int, a: int, shots: int = 512) -> int | None:
    n_bits = math.ceil(math.log2(N))
    m = 2 * n_bits
    qc = build_shor_order_circuit(N, a, m, n_bits)
    counts = run_counts(transpile(qc, backend_sim), shots=shots)
    for bitstr, _ in sorted(counts.items(), key=lambda item: item[1], reverse=True):
        y = int(bitstr, 2)
        r_cand = Fraction(y, 2 ** m).limit_denominator(N).denominator
        if r_cand > 1 and pow(a, r_cand, N) == 1:
            return r_cand
    return None

def shor_factor(N: int, bases: list[int]) -> tuple[tuple[int, int], int, int]:
    attempts = 0
    for a in bases:
        attempts += 1
        g = math.gcd(a, N)
        if 1 < g < N:
            return tuple(sorted((g, N // g))), a, attempts
        r = quantum_order_finding(N, a)
        if r is None or r % 2 != 0:
            continue
        x = pow(a, r // 2, N)
        if x == N - 1:
            continue
        for cand in (math.gcd(x - 1, N), math.gcd(x + 1, N)):
            if 1 < cand < N:
                return tuple(sorted((cand, N // cand))), a, attempts
    raise ValueError(f"Не удалось разложить N={N}")

factors, used_base, tries = shor_factor(35, [34, 2, 3])
p, q = factors
assert p * q == 35 and 1 < p < 35 and 1 < q < 35
assert {p, q} == set(sympy.factorint(35).keys())
print(f"[6] 35 = {p} * {q} (успешное основание a = {used_base}, попыток: {tries})")

### Ответы на контрольные вопросы (Задание 6)

1. **Простое число $N = 1009$:** алгоритм Шора никогда не вернёт нетривиальный делитель, так как у простого числа нет делителей, отличных от $1$ и $1009$, а в поле $\mathbb{Z}_{1009}$ единственными корнями уравнения $x^2 \equiv 1 \pmod{1009}$ являются $x \equiv \pm 1$.
2. **Нечётный порядок $r$:** разложение разности квадратов $a^r - 1 = (a^{r/2} - 1)(a^{r/2} + 1) \equiv 0 \pmod N$ требует целого показателя $r/2$, поэтому нечётный порядок $r$ не даёт целого $a^{r/2}$ и основание $a$ приходится менять.
3. **Число кубитов для 2048-битного $N$:** $3\lceil\log_2 N\rceil = 3 \cdot 2048 = \mathbf{6144}$ кубита (без учёта вспомогательных анцилл обратимой арифметики).